# トークンの文字位置（オフセット）を取得する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/token_offsets.ipynb)

NLP クックブック 03-07 ／ 日本語の形態素解析と文分割

各単語が元の文の何文字目から何文字目にあるかを知りたい（アノテーションとの対応付け、ハイライト表示など）。

In [1]:
!pip install -q sudachipy sudachidict_core fugashi unidic-lite

In [2]:
from sudachipy import Dictionary, SplitMode

tokenizer = Dictionary().create()
text = "東京都の 小池知事が会見した。"
for m in tokenizer.tokenize(text, SplitMode.C):
    print(m.begin(), m.end(), repr(m.surface()), repr(text[m.begin():m.end()]))

0 3 '東京都' '東京都'
3 4 'の' 'の'
4 5 ' ' ' '
5 7 '小池' '小池'
7 9 '知事' '知事'
9 10 'が' 'が'
10 12 '会見' '会見'
12 13 'し' 'し'
13 14 'た' 'た'
14 15 '。' '。'


In [3]:
# MeCab (fugashi) はオフセットを返さないので、表層形を前から順に探して復元する
from fugashi import Tagger

tagger = Tagger()

def with_offsets(text):
    pos, out = 0, []
    for w in tagger(text):
        start = text.index(w.surface, pos)
        pos = start + len(w.surface)
        out.append((start, pos, w.surface))
    return out

print(with_offsets(text))

[(0, 2, '東京'), (2, 3, '都'), (3, 4, 'の'), (5, 7, '小池'), (7, 9, '知事'), (9, 10, 'が'), (10, 12, '会見'), (12, 13, 'し'), (13, 14, 'た'), (14, 15, '。')]


- Sudachi は空白もトークンとして返すので、足し合わせると元の文に戻る。MeCab は空白を捨てる。
- 正規化をかけてから解析するとオフセットがずれる。先に正規化するなら、正規化後の文字列を「原文」として扱う。